# TapaHuecos: entrenamiento en Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vixanxva/TapaHuecos/blob/main/notebooks/entrenamiento_colab.ipynb)

Este notebook entrena el detector de daños (D00, D10, D20, D40) con RDD2022 en una GPU T4 gratuita y deja el
modelo en Google Drive como `TapaHuecos/modelos/best_rdd.pt`.

**Cómo usarlo:**

1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. Revisar la celda de configuración (paso 1).
3. *Entorno de ejecución → Ejecutar todas*. Colab pedirá permiso para acceder a Drive.

Todo lo importante (zips del dataset, pesos de cada época, gráficas y el modelo final) se guarda en Drive, así que
una desconexión de Colab no hace perder el entrenamiento: ver el paso 9.

## 1. Configuración

Es la única celda que normalmente hay que modificar.

- `MODO = "rapido"`: 1 o 2 países y `yolov8n.pt` durante 25 épocas. Sirve para tener pronto un modelo que funcione.
- `MODO = "completo"`: cinco países y `yolov8s.pt` durante 100 épocas (con parada temprana). Es el modelo del informe.
- `MODO = "prueba"`: un país pequeño y 1 época, solo para comprobar que el notebook funciona de principio a fin.

Los tiempos son aproximados: dependen de la carga de Colab y de la velocidad de descarga de figshare.

In [ ]:
# ---- Lo que normalmente se cambia ----
MODO = "rapido"  # "rapido", "completo" o "prueba"

# Países del modo rápido (1 o 2). Tamaños de cada zip: Czech 0,26 GB, India 0,53 GB, Japan 1,07 GB,
# United_States 0,44 GB, China_MotorBike 0,19 GB, China_Drone 0,16 GB (vista aérea), Norway 10,6 GB.
PAISES = ["India", "Japan"]

CARPETA_DRIVE = "/content/drive/MyDrive/TapaHuecos"

# Opcional: añadir las fotos de Montería etiquetadas en Roboflow (paso 7).
USAR_ROBOFLOW = False
ROBOFLOW_WORKSPACE = "nombre-del-workspace"
ROBOFLOW_PROYECTO = "nombre-del-proyecto"
ROBOFLOW_VERSION = 1

# ---- Valores de cada modo ----
# Tiempo total aproximado en una T4 (descarga + conversión + entrenamiento); estimaciones prudentes:
#   "prueba":   Czech, yolov8n, 1 época ................ unos 15 a 30 minutos.
#   "rapido":   India + Japan, yolov8n, 25 épocas ...... unas 1,5 a 3 horas.
#   "completo": 5 países, yolov8s, 100 épocas .......... unas 8 a 12 horas o más. No cabe en una sola
#               sesión gratuita: cuando Colab se desconecte, hay que reanudar con el paso 9.
if MODO == "rapido":
    MODELO_BASE, EPOCAS = "yolov8n.pt", 25
elif MODO == "completo":
    # Norway (10,6 GB) se deja fuera para no llenar los 15 GB gratuitos de Drive; China_Drone es vista aérea.
    PAISES = ["Japan", "India", "Czech", "United_States", "China_MotorBike"]
    MODELO_BASE, EPOCAS = "yolov8s.pt", 100
elif MODO == "prueba":
    PAISES = ["Czech"]
    MODELO_BASE, EPOCAS = "yolov8n.pt", 1
else:
    raise ValueError('MODO debe ser "rapido", "completo" o "prueba"')

DIR_DATOS = "/content/datos"  # disco local de Colab: rápido, pero se borra al desconectarse
DIR_REPO = "/content/TapaHuecos"
NOMBRE_RUN = f"rdd_{MODO}"
DESTINO_MODELO = f"{CARPETA_DRIVE}/modelos/best_rdd.pt"
print(f"Modo {MODO}: {MODELO_BASE}, {EPOCAS} épocas, países {PAISES}")

## 2. Verificar la GPU

Debe aparecer una Tesla T4. Si sale un error o el aviso de que no hay GPU: *Entorno de ejecución → Cambiar tipo de
entorno de ejecución → T4 GPU → Guardar* y volver a ejecutar desde el principio.

In [ ]:
!nvidia-smi

import torch

if torch.cuda.is_available():
    print("GPU disponible:", torch.cuda.get_device_name(0))
else:
    print("=" * 70)
    print("AVISO: NO HAY GPU. Entrenar en CPU tardaría días.")
    print("Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU")
    print("=" * 70)

## 3. Montar Google Drive

Se crean las carpetas del proyecto en Drive: `datos/` (zips de RDD2022), `runs/` (cada entrenamiento),
`modelos/` y `reportes/`.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
for subcarpeta in ("datos/RDD2022", "runs", "modelos", "reportes"):
    Path(CARPETA_DRIVE, subcarpeta).mkdir(parents=True, exist_ok=True)
print("Carpeta del proyecto en Drive:", CARPETA_DRIVE)

## 4. Clonar el repositorio e instalarlo

Si el repositorio ya estaba clonado en esta sesión, solo se actualiza.

In [ ]:
!test -d {DIR_REPO}/.git && git -C {DIR_REPO} pull --ff-only || git clone https://github.com/vixanxva/TapaHuecos.git {DIR_REPO}
%cd {DIR_REPO}
!pip install -q -e ".[yolo]"

## 5. Descargar RDD2022

En figshare, RDD2022 es un solo zip de unos 13,3 GB (`RDD2022_released_through_CRDDC2022.zip`) que contiene
**un zip por país** guardado sin comprimir. Por eso no hace falta bajar los 13,3 GB: la celda consulta la
[API de figshare](https://api.figshare.com/v2/articles/21431547), lee solo el índice del zip general y descarga
únicamente los bytes del zip de cada país de `PAISES`.

- Cada zip de país se guarda en `CARPETA_DRIVE/datos/RDD2022/` y, si ya está ahí, se reutiliza sin descargar nada.
- Se descomprime solo la partición `train` (la de `test` no trae etiquetas) en el disco local de Colab
  (`/content/datos/RDD2022`), porque leer las imágenes desde Drive durante el entrenamiento es muy lento.
- Si los zips internos vinieran comprimidos, la celda descarga el zip general completo y extrae de él los países.

**Alternativa manual**, si la descarga falla: bajar el zip desde
[la página de figshare](https://figshare.com/articles/dataset/RDD2022_-_The_multi-national_Road_Damage_Dataset_released_through_CRDDC_2022/21431547),
sacar de él los zips de los países (por ejemplo `RDD2022/India.zip`) y subirlos a `CARPETA_DRIVE/datos/RDD2022/`.
Al volver a ejecutar la celda, se reutilizan.

In [ ]:
import io
import json
import shutil
import struct
import time
import urllib.request
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

API_FIGSHARE = "https://api.figshare.com/v2/articles/21431547"
PAGINA_FIGSHARE = ("https://figshare.com/articles/dataset/RDD2022_-_The_multi-national_Road_Damage_Dataset_"
                   "released_through_CRDDC_2022/21431547")
DIR_RDD = Path(DIR_DATOS) / "RDD2022"
DIR_DESCARGAS = Path(DIR_DATOS) / "descargas"
DIR_ZIPS_DRIVE = Path(CARPETA_DRIVE) / "datos" / "RDD2022"


def pedir_rango(url, inicio, fin):
    """Pide los bytes [inicio, fin] de la URL; falla si el servidor no admite descargas parciales."""
    pedido = urllib.request.Request(url, headers={"Range": f"bytes={inicio}-{fin}"})
    respuesta = urllib.request.urlopen(pedido, timeout=60)
    if respuesta.status != 206:
        respuesta.close()
        raise RuntimeError("El servidor no admite descargas parciales.")
    return respuesta


class ArchivoRemoto(io.RawIOBase):
    """Archivo de solo lectura sobre HTTP: cada lectura pide un rango de bytes."""

    def __init__(self, url):
        self.url, self.pos = url, 0
        with pedir_rango(url, 0, 0) as respuesta:
            self.tamano = int(respuesta.headers["Content-Range"].split("/")[-1])

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, desplazamiento, desde=0):
        self.pos = (desplazamiento, self.pos + desplazamiento, self.tamano + desplazamiento)[desde]
        return self.pos

    def readinto(self, bufer):
        n = min(len(bufer), self.tamano - self.pos)
        if n <= 0:
            return 0
        with pedir_rango(self.url, self.pos, self.pos + n - 1) as respuesta:
            datos = respuesta.read()
        bufer[:len(datos)] = datos
        self.pos += len(datos)
        return len(datos)


def consultar_figshare():
    """Devuelve la URL de descarga y el tamaño del zip de RDD2022 según la API de figshare."""
    with urllib.request.urlopen(API_FIGSHARE, timeout=60) as respuesta:
        archivos = json.load(respuesta)["files"]
    for archivo in archivos:
        print(f"  figshare: {archivo['name']} ({archivo['size'] / 1e9:.2f} GB)")
    zips = [a for a in archivos if a["name"].lower().endswith(".zip")]
    if not zips:
        raise RuntimeError(f"La API de figshare no lista ningún zip. Descárguelo a mano: {PAGINA_FIGSHARE}")
    principal = max(zips, key=lambda a: a["size"])
    return principal["download_url"], principal["size"]


def indice_paises(url):
    """Lee solo el índice del zip general y ubica el zip de cada país: nombre, inicio y tamaño."""
    crudo = ArchivoRemoto(url)
    general = zipfile.ZipFile(io.BufferedReader(crudo, buffer_size=1 << 16))
    indice = {}
    for info in general.infolist():
        if not info.filename.lower().endswith(".zip"):
            continue
        crudo.seek(info.header_offset)
        cabecera = crudo.read(30)
        if cabecera[:4] != b"PK\x03\x04":
            raise RuntimeError(f"Cabecera inesperada en {info.filename}")
        largo_nombre, largo_extra = struct.unpack("<HH", cabecera[26:30])
        indice[Path(info.filename).stem] = {
            "nombre": info.filename,
            "inicio": info.header_offset + 30 + largo_nombre + largo_extra,
            "tamano": info.file_size,
            "sin_comprimir": info.compress_type == zipfile.ZIP_STORED,
        }
    if not indice:
        raise RuntimeError(f"El zip de figshare no trae zips por país. Revise {PAGINA_FIGSHARE}")
    return indice


def descargar_rango(url, inicio, tamano, destino):
    """Descarga `tamano` bytes desde `inicio`, con barra de progreso y reintentos que continúan donde iban."""
    destino.parent.mkdir(parents=True, exist_ok=True)
    parcial = destino.with_name(destino.name + ".parcial")
    hecho = parcial.stat().st_size if parcial.exists() else 0
    with tqdm(total=tamano, initial=hecho, unit="B", unit_scale=True, desc=destino.name) as barra:
        for _ in range(20):
            if hecho >= tamano:
                break
            try:
                with pedir_rango(url, inicio + hecho, inicio + tamano - 1) as respuesta, open(parcial, "ab") as f:
                    while bloque := respuesta.read(1 << 20):
                        f.write(bloque)
                        hecho += len(bloque)
                        barra.update(len(bloque))
            except OSError as error:
                print(f"Se cortó la descarga ({error}); se reintenta en 10 s...")
                time.sleep(10)
    if hecho != tamano:
        raise RuntimeError(f"No se pudo descargar {destino.name}. Alternativa manual: {PAGINA_FIGSHARE}")
    parcial.replace(destino)


_figshare = {}


def datos_figshare():
    """Consulta figshare una sola vez por sesión, y solo si hace falta descargar algo."""
    if not _figshare:
        url, tamano = consultar_figshare()
        _figshare.update(url=url, tamano=tamano, paises=indice_paises(url))
        print("  Países en el zip:", ", ".join(sorted(_figshare["paises"])))
    return _figshare


def descargar_pais(pais, destino):
    datos = datos_figshare()
    if pais not in datos["paises"]:
        raise ValueError(f"'{pais}' no está en RDD2022. Disponibles: {sorted(datos['paises'])}")
    entrada = datos["paises"][pais]
    if entrada["sin_comprimir"]:
        descargar_rango(datos["url"], entrada["inicio"], entrada["tamano"], destino)
        return
    # Caso de respaldo: el zip del país viene comprimido dentro del general y no se puede cortar por bytes.
    general = DIR_DESCARGAS / "RDD2022_completo.zip"
    if not (general.exists() and zipfile.is_zipfile(general)):
        print(f"Se descarga el zip completo ({datos['tamano'] / 1e9:.1f} GB); tardará bastante.")
        descargar_rango(datos["url"], 0, datos["tamano"], general)
    with zipfile.ZipFile(general) as z, z.open(entrada["nombre"]) as origen, open(destino, "wb") as copia:
        shutil.copyfileobj(origen, copia, 1 << 20)


def obtener_zip(pais):
    """Devuelve la ruta del zip del país: lo reutiliza de Drive o lo descarga y lo guarda en Drive."""
    en_drive = DIR_ZIPS_DRIVE / f"{pais}.zip"
    if en_drive.exists() and zipfile.is_zipfile(en_drive):
        print(f"{pais}: se reutiliza {en_drive}")
        return en_drive
    local = DIR_DESCARGAS / f"{pais}.zip"
    if not (local.exists() and zipfile.is_zipfile(local)):
        print(f"{pais}: descargando...")
        descargar_pais(pais, local)
    try:
        en_drive.parent.mkdir(parents=True, exist_ok=True)
        temporal = en_drive.with_name(en_drive.name + ".parcial")
        shutil.copyfile(local, temporal)
        temporal.replace(en_drive)
        print(f"{pais}: zip guardado en Drive ({local.stat().st_size / 1e9:.2f} GB)")
    except OSError as error:
        print(f"AVISO: no se pudo guardar {pais}.zip en Drive ({error}). ¿Hay espacio libre?")
    return local


def descomprimir(zip_pais, pais):
    """Extrae solo <pais>/train/ en DIR_RDD (aunque dentro del zip haya carpetas de más)."""
    xmls = DIR_RDD / pais / "train" / "annotations" / "xmls"
    if xmls.is_dir() and any(xmls.iterdir()):
        print(f"{pais}: ya estaba descomprimido")
        return
    marca = f"/{pais}/train/"
    with zipfile.ZipFile(zip_pais) as z:
        miembros = [m for m in z.namelist() if marca in f"/{m}"]
        if not miembros:
            raise RuntimeError(f"{zip_pais.name} no tiene la carpeta {marca}. Revise {PAGINA_FIGSHARE}")
        prefijo = f"/{miembros[0]}".split(marca)[0].strip("/")
        z.extractall(DIR_RDD, members=tqdm(miembros, desc=f"Descomprimiendo {pais}", unit="archivo"))
    if prefijo:  # p. ej. RDD2022/India/train/... -> India/train/...
        shutil.move(str(DIR_RDD / prefijo / pais), str(DIR_RDD / pais))


for pais in PAISES:
    descomprimir(obtener_zip(pais), pais)
    n_imagenes = len(list((DIR_RDD / pais / "train" / "images").glob("*.jpg")))
    print(f"{pais}: {n_imagenes} imágenes de entrenamiento en {DIR_RDD / pais}")

## 6. Convertir a formato YOLO

`scripts/preparar_rdd2022.py` conserva solo D00, D10, D20 y D40, guarda un 10 % de imágenes sin daños como
negativos y divide en entrenamiento, validación y prueba (80/10/10, con semilla fija). Tarda unos minutos.

In [ ]:
!rm -rf {DIR_DATOS}/rdd_yolo
!python scripts/preparar_rdd2022.py --origen {DIR_DATOS}/RDD2022 --destino {DIR_DATOS}/rdd_yolo --paises {" ".join(PAISES)}
DATA_YAML = f"{DIR_DATOS}/rdd_yolo/data.yaml"

## 7. Opcional: añadir las fotos de Montería (Roboflow)

Desactivado por defecto (`USAR_ROBOFLOW = False` en el paso 1). Para usarlo:

1. En Colab, abrir el panel **Secrets** (icono de la llave, a la izquierda) y crear el secreto `ROBOFLOW_API_KEY`
   con la API key de Roboflow. Activar el acceso del notebook a ese secreto.
2. Poner `USAR_ROBOFLOW = True` y los nombres del workspace, el proyecto y la versión en el paso 1.

**Nunca** escribir la API key en el notebook: quedaría en el historial de Git.

La celda descarga la exportación en formato YOLOv8 y la une con RDD2022 con `scripts/combinar_datasets.py`,
repitiendo 3 veces las fotos de Montería en entrenamiento.

In [ ]:
if USAR_ROBOFLOW:
    get_ipython().system("pip install -q roboflow")
    from google.colab import userdata
    from roboflow import Roboflow

    clave = userdata.get("ROBOFLOW_API_KEY")  # se lee de los Secrets de Colab
    proyecto = Roboflow(api_key=clave).workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROYECTO)
    proyecto.version(ROBOFLOW_VERSION).download("yolov8", location=f"{DIR_DATOS}/monteria")
    del clave
    get_ipython().system(f"rm -rf {DIR_DATOS}/combinado")
    get_ipython().system(
        f"python scripts/combinar_datasets.py --entradas {DIR_DATOS}/rdd_yolo {DIR_DATOS}/monteria "
        f"--destino {DIR_DATOS}/combinado --repetir 1 3"
    )
    DATA_YAML = f"{DIR_DATOS}/combinado/data.yaml"
print("Dataset de entrenamiento:", DATA_YAML)

## 8. Entrenar

Los resultados de cada época (`last.pt`, `best.pt`, `results.csv` y las gráficas) se escriben directamente en
`CARPETA_DRIVE/runs/<nombre>/`, así que no se pierden si Colab se desconecta. Al terminar, el mejor modelo se
copia a `CARPETA_DRIVE/modelos/best_rdd.pt`.

Si Colab se desconecta a mitad de camino, **no** volver a ejecutar esta celda (empezaría de cero): seguir el paso 9.

In [ ]:
import time

import torch

DISPOSITIVO = "0" if torch.cuda.is_available() else "cpu"
inicio = time.time()
!python scripts/entrenar.py --datos {DATA_YAML} --modelo {MODELO_BASE} --epocas {EPOCAS} --dispositivo {DISPOSITIVO} --proyecto {CARPETA_DRIVE}/runs --nombre {NOMBRE_RUN} --destino {DESTINO_MODELO}
print(f"Tiempo de entrenamiento: {(time.time() - inicio) / 60:.0f} min")

## 9. Reanudar si Colab se desconectó

Al desconectarse, Colab borra el disco local (el dataset), pero los pesos siguen en Drive. Para continuar:

1. Volver a conectar (con GPU) y ejecutar los pasos **1 a 6**. El paso 5 reutiliza los zips de Drive, así que es
   rápido, y el paso 6 genera exactamente la misma división (semilla fija). Si se usó Roboflow, ejecutar también el 7.
2. **Saltar el paso 8** y ejecutar esta celda: busca el `last.pt` más reciente en Drive y continúa desde esa época.

Si el entrenamiento ya había terminado, la celda solo copia su `best.pt` a `modelos/best_rdd.pt`.

In [ ]:
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

candidatos = sorted(Path(CARPETA_DRIVE, "runs").glob("*/weights/last.pt"), key=lambda p: p.stat().st_mtime)
if not candidatos:
    print("No hay ningún last.pt en Drive: no hay nada que reanudar.")
else:
    ultimo = candidatos[-1]
    punto = torch.load(ultimo, map_location="cpu", weights_only=False)
    if punto.get("epoch", -1) == -1:
        print(f"El entrenamiento de {ultimo.parents[1].name} ya había terminado; no hay nada que reanudar.")
    else:
        print(f"Se reanuda {ultimo} desde la época {punto['epoch'] + 2}")
        YOLO(str(ultimo)).train(resume=True)
    mejor = ultimo.with_name("best.pt")
    if mejor.exists():
        Path(DESTINO_MODELO).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(mejor, DESTINO_MODELO)
        print("Mejor modelo copiado a", DESTINO_MODELO)

## 10. Evaluar

`scripts/evaluar.py modelo` calcula precisión, exhaustividad, F1 y mAP por clase sobre la partición de prueba y
guarda la tabla en `CARPETA_DRIVE/reportes/`. Después se muestran las gráficas del entrenamiento, que van al
informe y a la issue #9.

In [ ]:
!python scripts/evaluar.py --salida {CARPETA_DRIVE}/reportes modelo --pesos {DESTINO_MODELO} --datos {DATA_YAML}

In [ ]:
from pathlib import Path

from IPython.display import Image, display

corridas = sorted(Path(CARPETA_DRIVE, "runs").glob("*/results.png"), key=lambda p: p.stat().st_mtime)
if not corridas:
    print("No se encontró ninguna corrida con results.png en", Path(CARPETA_DRIVE, "runs"))
else:
    carpeta = corridas[-1].parent
    print("Corrida:", carpeta)
    # Según la versión de Ultralytics, la curva PR se llama PR_curve.png o BoxPR_curve.png.
    for patron in ("results.png", "confusion_matrix_normalized.png", "*PR_curve.png"):
        encontradas = sorted(carpeta.glob(patron))
        if encontradas:
            print(encontradas[0].name)
            display(Image(filename=str(encontradas[0]), width=900))
        else:
            print(f"No se encontró {patron} en {carpeta}")

## 11. Prueba rápida

Predice sobre algunas imágenes de la partición de prueba (que el modelo no vio al entrenar) y las muestra con
sus recuadros. Para ver otras imágenes, cambiar la semilla.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
from ultralytics import YOLO

SEMILLA = 0
modelo = YOLO(DESTINO_MODELO)
imagenes = sorted(Path(DIR_DATOS, "rdd_yolo", "images", "test").glob("*.jpg"))
muestra = random.Random(SEMILLA).sample(imagenes, min(6, len(imagenes)))
resultados = modelo.predict([str(p) for p in muestra], conf=0.25, verbose=False)

figura, ejes = plt.subplots(2, 3, figsize=(18, 11))
for eje in ejes.flat:
    eje.axis("off")
for eje, resultado, ruta in zip(ejes.flat, resultados, muestra):
    eje.imshow(resultado.plot()[:, :, ::-1])  # plot() devuelve BGR
    clases = ", ".join(resultado.names[int(c)] for c in resultado.boxes.cls) or "sin detecciones"
    eje.set_title(f"{ruta.name}\n{clases}", fontsize=9)
plt.tight_layout()
plt.show()

## 12. Usar el modelo en el computador

El modelo queda en Drive: `MyDrive/TapaHuecos/modelos/best_rdd.pt`.

1. Descargarlo desde [drive.google.com](https://drive.google.com) (o con la celda de abajo) y copiarlo a la carpeta
   `modelos/` del repositorio local.
2. Procesar un recorrido con ese modelo:

```bash
tapahuecos procesar --video datos/videos/VID_20261029_143015.mp4 --gpx datos/gpx/20261029.gpx --pesos modelos/best_rdd.pt
```

Para no tener que pasar `--pesos` cada vez, se puede poner esa ruta en `modelo.pesos` de `config.yaml`.
Recordar anotar en la issue #9 el mAP@0.5, el número de épocas, el tiempo total y adjuntar las gráficas.

In [ ]:
from google.colab import files

files.download(DESTINO_MODELO)  # descarga directa al navegador (opcional)